In [ ]:
import sys

if "google.colab" in sys.modules:
    import os

    repo = "/content/lmbike_workshop_gimbal"
    sim = "/content/lmbike_workshop_gimbal/bike_mujoco_sim"
    if not os.path.isdir(repo):
        !git clone https://github.com/innowing-robotics-interns/lmbike_workshop_gimbal.git /content/lmbike_workshop_gimbal
    %cd /content/lmbike_workshop_gimbal/bike_mujoco_sim
    if sim not in sys.path:
        sys.path.insert(0, sim)

    # Colab's Jupyter must stay the copy it shipped with. requirements.txt
    # does not list Jupyter. The constraint pins whatever Jupyter packages
    # are already installed so pip cannot replace them.
    import importlib.metadata as metadata
    protected = {
        "notebook", "jupyter", "jupyterlab", "jupyter-client", "jupyter-core",
        "jupyter-server", "ipykernel", "ipython", "nbformat", "nbclient",
        "nbconvert", "nbclassic",
    }
    pinned = []
    for dist in metadata.distributions():
        name = dist.metadata["Name"]
        if name and name.lower().replace("_", "-") in protected:
            pinned.append(f"{name}=={dist.version}")
    with open("/tmp/colab-jupyter-constraints.txt", "w", encoding="utf-8") as fh:
        fh.write("\n".join(pinned) + "\n")
    !pip install -q --upgrade-strategy only-if-needed -r requirements.txt -c /tmp/colab-jupyter-constraints.txt


# Lesson 1. The environment and the fall

Welcome to the first lesson. Over this series you will teach a computer to keep a bicycle upright. A person solves this problem without thinking every time they ride, but a computer has to be told exactly what to do. We will not start with equations. In this lesson you will watch a simulated bicycle ride for a few seconds while a computer program tries to balance it. Then you will look at a graph of how far the bicycle leaned during that ride. Seeing the bicycle succeed or fail with your own eyes first will make every later idea much easier to picture.

**Before you start.** If the first code cell stops with a message like `No module named …`, or VS Code says it cannot find Python, your computer is not set up yet. Open `00_setup_and_install.ipynb` in this same `tutorials` folder, follow it to the end, and then come back here.

**How to use this notebook.** Work from top to bottom, in order. There are two kinds of cells:

- **Text cells**, like this one, are for reading.
- **Code cells** have `[ ]` on their left. These are small programs that you run. To run one, click on it and press **Shift+Enter**.

While a code cell is running, its brackets show `[*]`. When it finishes, a number appears inside them, such as `[3]`. Wait for that number before you move on, because later cells depend on the earlier ones having finished.

## Learning objectives

By the end of this lesson you will be able to:

1. Describe what the physics engine does during the ride, and which job it leaves to the controller.
2. Read the bicycle's planned speed from its settings, and explain why moving forward alone does not keep the bicycle upright.
3. Name the three parts of a PID controller, and point to the one setting this lesson changes.
4. Connect the ride you watched in the 3D window with a graph of lean against time.

## Glossary

This section introduces the handful of ideas the whole series is built on. Each one starts with an everyday picture. Read it once now. The later steps use these words freely, and you can scroll back here whenever one of them feels unclear.

### Force and motion: F = ma

A push or a pull is called a **force**, written F. How much matter an object contains is its **mass**, m. How quickly the object's speed changes is its **acceleration**, a. The rule F = ma ties these three together. If you push the same object harder, it speeds up faster. If you push a heavier object just as hard, it speeds up more slowly.

Picture a shopping cart. An empty cart shoots forward with a light push. The same push on a cart full of groceries barely gets it rolling. That is F = ma in everyday life.

### The physics engine

A **physics engine** is a program that applies rules like F = ma over and over, very quickly, to work out how objects move. Video games use the same kind of software to make things fall, bounce and roll realistically. In this course the physics engine is called **MuJoCo**.

At each tiny time step, MuJoCo looks at every force on the bicycle, the wheels and the ground. These include gravity, the ground pushing back up, and the motors twisting. It then moves everything forward a little. It repeats this about 2,000 times for every second of the ride, which is fast enough to look like smooth, real motion. The 3D window you will open in Step 3 is MuJoCo showing you that motion.

Keep one thing in mind: the physics engine only works out *what happens*. It does not decide how to steer. Deciding is the controller's job, which is explained below.

### Free-body diagram

A **free-body diagram** is a simple sketch of one object, with an arrow for every force acting on it. Picture a sled being pulled across snow. You would draw the sled, then an arrow forward for the rope, an arrow down for gravity, and an arrow up for the snow holding the sled up. Engineers draw these diagrams to make sure they have not forgotten any force.

For our bicycle, the diagram has gravity pulling down, the ground pushing up through the tires, and two motors twisting. One motor drives the rear wheel to move the bicycle forward. The other turns the handlebars to steer. The physics engine is, in effect, working through this diagram at every time step.

### A spring and its stiffness: F = −kx

Stretch a spring and it pulls back. Stretch it farther and it pulls back harder. The rule F = −kx describes this:

- x is how far you have stretched the spring.
- k is the spring's **stiffness**. A stiff spring (big k) pulls back hard even for a small stretch. A soft spring (small k) pulls back gently.
- The minus sign means the pull always points back toward where the spring started, the opposite way to the stretch.

Think of a door with a spring closer at the top. Open it a little and it swings back gently. Open it wide and it swings back hard. The farther you push the door from closed, the harder it pulls itself back.

### Gain

The k in F = −kx is an example of a **gain**. A gain is a number that decides how strongly something reacts to what it measures. The measurement goes in, it is multiplied by the gain, and the result is the reaction.

Here is an example. Imagine adjusting a shower that is too cold, using a simple rule: for every degree the water is too cold, turn the hot tap by a fixed number of notches. That fixed number is your gain.

- With a gain of 1, water that is 3 degrees too cold makes you turn the tap 3 notches.
- With a gain of 5, the same 3 degrees makes you turn the tap 15 notches.

The water was equally cold both times. Only the gain changed, and the bigger gain gave a much stronger reaction.

The balancing program in this course has several gains. This lesson changes just one of them, named `pid_kd_roll`. The PID section below explains what it does.

### How a bicycle stays up

A bicycle standing still on its own falls over. A rider stays up because they keep making small corrections with the handlebars. When the bicycle starts to lean to the right, the rider turns the handlebars slightly to the right. This steers the wheels back underneath the rider's weight, and the bicycle comes back upright.

You can feel the same idea by balancing a broom upright on the palm of your hand. When the broom starts to tip to the left, you move your hand to the left, back underneath it. Your eyes watch the tilt, and your hand moves the bottom of the broom to catch it. On a bicycle, steering is how the bottom, the wheels, gets moved.

Steering only works this way while the bicycle is rolling forward. A wheel that is standing still can be turned left and right all day without moving anywhere.

So the job in this course can be said in one sentence: **measure how far the bicycle is leaning, and turn the handlebars to catch it.** The program that does this is called the controller.

### Controller and balancing loop

A **controller** is the part of a program that makes decisions. It reads measurements, such as how far the bicycle is leaning, and sends commands, such as how to turn the handlebars. It does this over and over in a cycle: measure, decide, command, and then measure again to see what happened. Because it goes round and round, this cycle is called a **control loop**. In this course, the control loop that keeps the bicycle upright is called the **balancing loop**.

Keep the two roles separate in your mind. The physics engine is the world: it moves the bicycle according to the forces on it. The balancing loop is the rider: it decides how to steer.

### PID

The balancing loop in this course is a **PID controller**. PID is short for **Proportional, Integral, Derivative**. These are three different ways of looking at the bicycle's lean. The goal is always the same: an upright bicycle, which is a lean of 0°. Each of the three parts works out its own amount of steering, and the three amounts are added together into one handlebar command.

- **Proportional (P)** looks at how far the bicycle is leaning *right now*. A small lean gets a small correction and a big lean gets a big correction, just like the spring that pulls harder the farther it is stretched.
- **Integral (I)** looks at *how long* a lean has lasted. If the bicycle stays tilted a little to one side for a while, this part slowly adds more and more correction until the tilt is gone.
- **Derivative (D)** looks at *how fast* the lean is changing. If the bicycle is tipping over quickly, this part reacts strongly straight away. It acts like a brake on the fall before the fall gets big.

Each part has its own gain, which says how strongly that part reacts:

- `pid_kp_roll` is the gain for P.
- `pid_ki_roll` is the gain for I.
- `pid_kd_roll` is the gain for D.

The word `roll` at the end of each name means these gains react to the bicycle's lean. Roll is explained in the table below. This lesson changes only the D gain, `pid_kd_roll`. Lesson 2 goes through all three parts carefully, with numbers from the bicycle. From here on, we will often just say P, I and D.

### Other words you will meet

| Term | Meaning | Everyday example |
| --- | --- | --- |
| Simulation | The physics engine working out the motion in many tiny time steps, then playing them back fast enough to look like a real ride. | A flip book: many still drawings, shown quickly, look like movement. |
| Model | The description of one bicycle inside the simulation: how heavy each part is, how big the wheels are, and how the parts are joined. This lesson uses the model named `new_bike`. | The bicycle you would draw in the free-body diagram. |
| Roll | How far the bicycle is leaning sideways, in degrees. 0° is perfectly upright, and 90° is lying flat on the ground. A positive number is a lean to one side, and a negative number is a lean to the other side. | A book standing on its edge: as it starts to tip over, the angle it has tipped is its roll. |
| Torque | A twisting push. Motors produce torque. The steering motor's torque is what turns the handlebars. | Your hand twisting a doorknob or a jar lid. |
| Configuration | The list of numbers, such as speeds, times and gains, that describes one ride. The numbers are stored in a text file whose name ends in `.yaml`. | The settings on a phone. The phone stays the same, but changing the settings changes what it does. |
| Reference | A target value that we ask for. The bicycle may not reach it exactly. In the code, the names of targets end in `_ref`. | The temperature you set on an oven. The oven is asked to reach it, but the food is not that hot yet. |

## Step 1. Start the lesson

The first code cell prepares the notebook. It finds the project folder on your computer, so that the lesson can load the bicycle and its settings, and it loads the tools used to draw graphs. You only need to run it. Reading the code inside is optional.

Run the next cell now. When it finishes, look for the line that begins with `project_root:`. It shows the folder this lesson is working from, and it should end in `bike_mujoco_sim`.

If the cell instead says it cannot find `bike_mujoco_sim`, Jupyter was started from the wrong folder. Close Jupyter, start it again from the folder that contains `run_sim.py` (or from its `tutorials` folder), and then run this cell again.

In [ ]:
import os
import sys

def choose_mujoco_gl(platform, display, existing):
    if existing:
        return existing
    if str(platform).startswith("linux") and not display:
        return "egl"
    return None

chosen = choose_mujoco_gl(sys.platform, os.environ.get("DISPLAY"), os.environ.get("MUJOCO_GL"))
if chosen and "MUJOCO_GL" not in os.environ:
    os.environ["MUJOCO_GL"] = chosen

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import replace

from sim.config import load_config
from sim.loop import make_sim, run_headless, summarize
from sim.pid_controller import RollSteerPID, apply_speed_refs

np.set_printoptions(precision=4, suppress=True)
print("project_root:", project_root)
print("platform:", sys.platform, "MUJOCO_GL:", os.environ.get("MUJOCO_GL", "(unset)"))


## Step 2. Look at how fast the bicycle is asked to go

As you read in the glossary, steering can only balance a bicycle that is rolling forward. So before we watch the ride, let's look at how fast the bicycle will be asked to go.

The program does not make the bicycle jump straight to full speed. It follows a **speed schedule**: a plan, written down before the ride starts, of what the bicycle's forward speed should be at each moment. Our schedule is simple:

1. The bicycle starts at rest, at 0 m/s.
2. Over the first `acceleration_time` seconds, it speeds up smoothly, like a car pulling away from traffic lights.
3. After that, it holds one steady speed, `target_speed`, for the rest of the ride. We call that steady speed the **cruise speed**.

The motor on the rear wheel is what pushes the bicycle forward to follow this schedule.

We inspect the schedule to see clearly what the bicycle is being asked to do, and what it is not. The schedule only talks about going *forward*. It says nothing at all about leaning *sideways*. Think of cruise control in a car: it keeps the car at a set speed, but it does nothing to keep the car in its lane. In the same way, the speed schedule moves the bicycle forward, and keeping it upright is left entirely to the balancing loop.

The next cell reads the settings file `configs/new_bike.yaml`. It then asks the program for its targets at a few moments during the ride. For each moment it prints three references. Remember that a reference is a target we ask for, not a measurement.

- `speed_ref` is the forward speed we ask for, in meters per second.
- `roll_ref` is the lean we ask for, in degrees. 0 means upright.
- `steer_ref` is the handlebar angle we ask for. 0 means pointing straight ahead.

In the code you will see `speed_goal=None`. This is a technical detail. It tells the program "nobody is driving with a joystick, so follow the speed schedule". It does not mean a speed of zero.

Run the next cell. Then check three things in the printout:

1. `speed_ref` starts at 0, grows over the first few lines, and then stays at the value printed next to `target_speed`.
2. `roll_ref` is 0 on every line. We always ask for an upright bicycle.
3. `steer_ref` is 0 on every line. We always ask for straight handlebars.

Keep in mind that these numbers are only what we *ask for*. Whether the bicycle actually stays upright is what you will find out in the next step.

In [ ]:
new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
cfg = load_config(new_bike_yaml)
print("target_speed", cfg.target_speed, "m/s")
print("acceleration_time", cfg.acceleration_time, "s")
for t in (0.0, 0.5, 1.0, cfg.acceleration_time, cfg.acceleration_time + 1.0):
    speed_ref, roll_ref, steer_ref = apply_speed_refs(
        mode="speed-schedule",
        time_s=t,
        speed_goal=None,
        steer_ref=0.0,
        cfg=cfg,
    )
    print(f"t={t:4.2f}s  speed_ref={speed_ref:.3f}  roll_ref={roll_ref}  steer_ref={steer_ref}")


## Step 3. Let's watch the bicycle in action

Now it is time to watch a ride. The bicycle will follow the speed schedule from Step 2. The balancing loop will be switched on for the whole ride, trying to keep the bicycle upright by steering.

For this ride, one gain has been deliberately set very low. The D gain, `pid_kd_roll`, is set to 1.

Here is why that matters. Recall that the D part reacts to *how fast* the bicycle is tipping. Picture two bicycles that are both leaning 5° at this instant. The first is drifting over slowly. The second is falling quickly. The P part only sees "5°", so it gives both bicycles the same correction. The D part sees that the second bicycle is tipping much faster, and adds a much stronger correction for it, like braking harder on a fall that is getting away. With the D gain at 1, that brake is weak. Watch what the bicycle does with a weak brake.

When you run the cell, it first prints the full configuration for this ride, so that you can see every number the program is using. Here is what each line means:

| Name | What it means |
| --- | --- |
| `file` | The settings file the numbers came from. |
| `target_speed` | The cruise speed, in m/s, that the bicycle holds after it has sped up. |
| `acceleration_time` | How many seconds the speeding up takes. |
| `duration` | How many seconds the whole ride lasts. |
| `pid_kp_roll` | The P gain: how strongly to steer against the lean the bicycle has right now. |
| `pid_ki_roll` | The I gain: how strongly to steer against a lean that has lasted for a while. |
| `pid_kd_roll` | The D gain: how strongly to steer against a lean that is changing fast. It is set to 1 for this ride. |
| `pid_kp_steer` | A small extra gain that gently pulls the handlebars back toward pointing straight ahead, so that they do not drift off to one side. Lesson 2 explains it. |
| `steer_kp`, `steer_ki` | The gains of a second, smaller control loop inside the steering motor. The balancing loop decides how fast the handlebars should turn, and this inner loop works out how much motor torque that takes. Keep these two at their printed values for the whole course. |

To run the ride:

1. Run the next cell and read the printed configuration. Check that `pid_kd_roll` shows 1.
2. A 3D window opens and the ride plays. Watch how the bicycle moves, and especially how it leans.
3. When the ride is over, close the MuJoCo window. The cell keeps running until the window is closed, so its number only appears after you close it.

If the cell stops with a message saying MuJoCo was already imported, open the **Kernel** menu and choose **Restart Kernel**. This clears the notebook's memory of the earlier cells. Run this cell again, and then run the Step 1 cell once more before you continue to Step 4.

In [ ]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
cfg = load_config(new_bike_yaml)
cfg = replace(cfg, pid_kd_roll=1.0, duration=4.0)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)
print("configuration for this ride")
print(f"  file                 {os.path.relpath(new_bike_yaml, project_root)}")
print(f"  target_speed         {cfg.target_speed} m/s")
print(f"  acceleration_time    {cfg.acceleration_time} s")
print(f"  duration             {cfg.duration} s")
print(f"  pid_kp_roll          {cfg.pid_kp_roll:.4f}")
print(f"  pid_ki_roll          {cfg.pid_ki_roll:.4f}")
print(f"  pid_kd_roll          {cfg.pid_kd_roll:.4f}  (set to 1 for this ride)")
print(f"  pid_kp_steer         {cfg.pid_kp_steer:.4f}")
print(f"  steer_kp             {cfg.steer_kp:.4f}")
print(f"  steer_ki             {cfg.steer_ki:.4f}")
print("Close the MuJoCo window to finish this cell.")
run_viewer(sim)
print("Window closed.")


## Step 4. Draw the lean as a graph

Watching the window shows you roughly what happened, but it is hard to judge exact angles and times by eye. A graph makes that easy.

The next cell runs exactly the same ride again, with the same settings, but this time without the 3D window. While it runs, it records the bicycle's lean, its roll, many times a second. When the ride is over, it draws those recordings as a line.

Run the next cell. When the graph appears, continue to the text below it, which walks you through how to read it.

In [ ]:
new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
cfg = load_config(new_bike_yaml)
cfg = replace(cfg, pid_kd_roll=1.0, duration=4.0)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)
print(f"same ride as the window, pid_kd_roll = {cfg.pid_kd_roll:.4f}")
history = run_headless(sim, duration=4.0)

time_s = np.asarray(history["time"], dtype=float)
roll_deg = np.degrees(np.asarray(history["roll"], dtype=float))
max_abs = float(np.max(np.abs(roll_deg)))
tip = np.where(np.abs(roll_deg) > 45.0)[0]
tip_t = float(time_s[tip[0]]) if len(tip) else None

fig, ax = plt.subplots(figsize=(8, 3.5), constrained_layout=True)
ax.plot(time_s, roll_deg)
ax.axhline(45.0, color="0.5", linestyle="--", linewidth=0.8)
ax.axhline(-45.0, color="0.5", linestyle="--", linewidth=0.8)
if tip_t is not None:
    ax.axvline(tip_t, color="C3", linestyle=":", label="lean reaches 45°")
    ax.legend()
ax.set_xlabel("time (s)")
ax.set_ylabel("roll (deg)")
ax.set_title("pid_kd_roll = 1")
plt.show()

if max_abs < 45.0:
    raise RuntimeError("expected this ride to fall")


## Reading the graph

The graph is the ride you just watched, turned into a line. Here is how to read it:

- **Left to right is time.** The left edge is the moment the ride started, and each step to the right is later in the ride. The numbers along the bottom are seconds.
- **Up and down is lean.** The height of the line at any moment is how far the bicycle was leaning at that moment, in degrees.
- **The 0° level means upright.** If the bicycle stayed perfectly balanced, the line would run flat along 0°. A line that stays within a few degrees of 0° belongs to a bicycle that stayed nearly upright.
- **Above or below 0° tells you the side.** A positive number is a lean to one side and a negative number is a lean to the other side. The farther the line is from 0°, the more the bicycle leaned.

The graph also has two dashed horizontal lines, at +45° and −45°. This course uses them to decide whether a ride counts as a fall. Once a bicycle leans past 45°, it is too far over to recover, so if the line crosses either dashed line, we say the bicycle fell. The dashed lines are measuring marks, like the marks on a ruler. They are not a path the bicycle was asked to follow. The request in Step 2 was always `roll_ref = 0`, which is the 0° level.

If the line does cross a dashed line, a dotted vertical line appears at the first moment that happened. Read straight down from it to the time axis to see how many seconds into the ride the fall happened.

If the line goes beyond 90°, the bicycle has gone past lying on its side. Remember that 90° is lying flat, so a value such as 117° means the bicycle has toppled over completely and is lying on the ground.

Take a moment to compare the graph with what you saw in the window. They should tell the same story, because they are the same ride.

## Questions

These questions check two things: whether the graph and the window told you the same story, and whether you can predict what a different D gain will do. Write down your answers before you run the next cell. A prediction only teaches you something if you make it before you see the result.

1. In Step 2, the program asked for `roll_ref = 0`, an upright bicycle, at every moment. Now look at the graph. Did the bicycle actually stay at 0°? If not, roughly when did it stop being close to upright?
2. This ride used a D gain of `pid_kd_roll = 1`, with every other gain left as printed. Imagine the same ride with a much larger D gain, `pid_kd_roll = 11`, and every other gain unchanged. Remember that the D part is the brake on a fast fall. Do you predict that the lean will cross 45°, or stay under 45°? Why?

Now test your prediction with the next cell. In that cell, `kd_try` is a short name for the D gain, `pid_kd_roll`. The cell runs the ride without a window using whatever number you put in `kd_try`, and prints a short report:

- the largest lean the bicycle reached during the ride, and
- either the time when the lean crossed 45°, or a sentence saying it stayed under 45°.

Try these three values in order. Each time, edit the number after `kd_try =` and run the cell again:

1. `1.0`, the value you just watched. The report should match your graph.
2. `4.0`, a moderately stronger brake.
3. `11.0`, a much stronger brake.

After each run, compare the result with your prediction.

In [ ]:
# kd_try is pid_kd_roll, the derivative gain.
# Edit kd_try and run again. Trials: 1.0, then 4.0, then 11.0
kd_try = 1.0

new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
cfg = load_config(new_bike_yaml)
cfg = replace(cfg, pid_kd_roll=float(kd_try), duration=4.0)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)
history = run_headless(sim, duration=4.0)
time_s = np.asarray(history["time"], dtype=float)
roll_deg = np.degrees(np.asarray(history["roll"], dtype=float))
max_abs = float(np.max(np.abs(roll_deg)))
tip = np.where(np.abs(roll_deg) > 45.0)[0]
tip_t = float(time_s[tip[0]]) if len(tip) else None
print(f"pid_kd_roll = {cfg.pid_kd_roll:.4f}")
print(f"largest lean = {max_abs:.1f} degrees")
if tip_t is None:
    print("Result: the lean stayed under 45°.")
else:
    print(f"Result: the lean passed 45° at {tip_t:.2f} s.")


## Discussion

Here are some questions students often ask after this lesson, with explanations.

### Why did the bicycle tip over even though a balancing loop was running?

It is natural to expect that switching on a balancing loop is enough to keep the bicycle up. But a controller is only as good as its gains. In this ride the loop was on from the first moment to the last, but its D gain was only 1, so its brake on a fast fall was weak. When the bicycle started tipping, the loop did steer against it, just not strongly enough or soon enough. The tip grew faster than the steering could catch it.

If you tried the larger values in the Questions, you saw that the very same loop, with only the D gain changed, behaves very differently.

### Why doesn't moving forward keep the bicycle up by itself?

Forward speed is needed, because steering only helps while the wheels are rolling. But speed on its own does nothing to correct a lean. The rear wheel pushes the bicycle *forward*. It never pushes it sideways, back toward upright.

Asking for `roll_ref = 0` does not help on its own either, because a reference is only a request. Something has to turn that request into steering, and that something is the balancing loop. So a moving bicycle with a weak balancing loop still falls, which is exactly what the graph showed.

### Does the size of one gain really matter that much?

Yes. In the Questions you kept the bicycle, the speed schedule and every other gain the same, and changed only one number, `pid_kd_roll`. That single number decided whether the ride ended in a fall.

Choosing gains well is a central part of control engineering, and it is what the next lessons are about. Lesson 2 opens up the balancing loop and shows, with numbers, how P, I and D each produce their part of the steering command.